# Top-k and top-p sampling

Implement the next-token sampler of a language model.

```python
def filter_logits(logits, temperature=1.0, top_k=0, top_p=1.0):
    # Your code: returns probabilities, same shape as logits

def sample(logits, generator, temperature=1.0, top_k=0, top_p=1.0):
    # Your code: returns token ids, shape (B,)
```

`logits` is a float tensor of shape `(B, V)`, one row per sequence. `filter_logits` applies, in this order:

1. **Temperature:** divide the logits by `temperature` ($> 0$).
2. **Top-k:** if `top_k > 0`, keep the `top_k` largest logits of each row and give every other token probability 0. (Rows in the tests have no ties.)
3. **Softmax** over each row.
4. **Top-p:** if `top_p < 1`, sort each row's probabilities in decreasing order and keep the smallest set of most likely tokens whose total probability is at least `top_p` (the token that crosses `top_p` is kept); every other token gets probability 0. Then renormalize the kept probabilities to sum to 1.

It returns the resulting probabilities, shape `(B, V)`, in the original token order.

`sample` draws one token per row from `filter_logits(...)` using `torch.multinomial` with the given `torch.Generator`, so that the same generator seed always gives the same tokens and the global random state is not used. It returns an `int64` tensor of shape `(B,)`.

Do not modify `logits`.

**Example.**

```python
z = torch.tensor([[2.0, 1.0, 0.0, -1.0]])
filter_logits(z, temperature=0.5)      # tensor([[0.8650, 0.1171, 0.0158, 0.0021]])
filter_logits(z, top_k=2)              # tensor([[0.7311, 0.2689, 0.0000, 0.0000]])
filter_logits(z, top_p=0.9)            # tensor([[0.6652, 0.2447, 0.0900, 0.0000]])
g = torch.Generator().manual_seed(0)
sample(z, g, top_k=2).shape            # torch.Size([1])
```

(For `top_p=0.9` the sorted probabilities are 0.6439, 0.2369, 0.0871, 0.0321; the running total first reaches 0.9 at the third token, so three tokens are kept and renormalized.)

**Restrictions.** Only `import torch`. No loops or comprehensions.

**Constraints.** $B \le 64$, $V \le 50{,}000$. Time limit 1 second per call on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem3_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem3_submission.py
import torch


def filter_logits(logits, temperature=1.0, top_k=0, top_p=1.0):
    # Your code
    return probs


def sample(logits, generator, temperature=1.0, top_k=0, top_p=1.0):
    # Your code
    return tokens


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem3_submission as submission
importlib.reload(submission)
# e.g. submission.filter_logits(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwtz2za2/itYdWZDpjIj27HbKHHudVq3mzt5TZKd3R3Jw0IiJHNNkSxBxXa8/u/3OwcACVKyE6e9t52JKRA4L5w3wOtBmgzGYrDePdwpR/uDoRisVC0xdL35pk7rTNHYx6LcORcyT0SNp1JouSqzNF/SpExpXeRuZTZ6zDDTfF0rjdHdA/xM0sUina+z+goj+xhQeV3R82CRZrWq4qxYprXmlUWyzlScyxVjLqtilqnVfqzXs1WqdQpUmCWzrLhQSZyuyqKqCc9kUBfV/GxwirczmeeKmJnYX4BflDSrrtaKGFO6jut0pYp1HdP43ohG5dJAylYR+IzPmT3zXLpntSpVJet1pVhChDNagbM0L1apzBi/LtbVnMkvqnSZ5hjGaCmres/ScDNk1llC14MzWeWQYlSySKb8/99UBkRa1GeyFlYKYlnJhAYN2yL4bVEVK2GXu9GHv4XRNJ/mx3a6SLWYF3kNmT9a5xDzo2f05/kjCxW/yzR5/sjMBhGReFmLRC1SQBUkKrFY5/MaoteC9iXhwfihoa2W5wpPapq3WyTMLorfMPRbJE7+efz63auTDyJLdU0sKQOWgGmhz4qLXGDR+3Uugk+pTkFV+FR8IHC1qNZAqz6p6ooXgbVfZJpB/lpUMtVK0E9xkdZnQgrA03KpxEWV1mBYLIpKvFD5GBp7JS6I3Aup8bbIl0ORE1SRrFelOEuTBNPTvFzXmqRndiG3Ip0X5VXzg/SG5Gt/5lh/JQA1L2n0O/FTkcmZ+Ond37WQlRKaNLUiWeVEygMtXu+Jd1UxFLoQWTGXGUGEMWmxWkMoM0VChYgKshmoFKRGcpiB/zNs5ULOoXURYdqLRrS5UizXtP+BwawBK1M79VmlZCJ0qbBhmFUV6+VZdgVtyRYwZaJiXlQk6EqR0GuRLohA/GSbsnxYKyGhfHh3cvJz/MvxTx/fvhdHhJ0YnubzTAI7bUNwcjlXJWlKOJ7mAv+VeGVmQaEgcZUE0MVkKFZ66eYAb16QkPPEjtB/7eYGNLcFEudlcOnWwo14a9wOkVV6owvwn+a6lvlcBZdD8z76qHJdVKG3nNEq2HYuLqMEbnF+FoTRvFzjX97mIDSTFbMpXjK6k6oqKg+KYdmDlZeR1LKq5BXobvnQZ7JUwbKoh1DKnP6Ffh5NB9Bs+JPpwFGmIet6XWYqIM4xP4x4qaWFZYo5bhLBCodiMR1cE8AbbLg2uMS1ht9Rl6Wa19CJa2/+DdC1lM2zQt9F2VBUdZEd7aqdw6GQ9vFHRy8s6wIkN8RKXV+B0UVWSCKMxhln94XHzNLwRyxdmMdb2DEM2OnhNt7cK2avQYANSTXcW1pjNejIsiAU8BSkhf67C/vOR0+eVKbsCXNak+aLBrhVZNrvLLNChDCsuOgfKy76x1c8VVWQGAuCiFvJS/ojZzpYih1xEYYhwQaj6WclVAa7GJHxbTGVrpzY0YlPMoN/EEEmqyW5XUUKK67xZxztL2/Czt6r39cy+wqt7BqVmUpE5wlr+jBkiYIIWdeVfT8d8Ha0QLaoi6cg7Zz/c7XwVYPpj60cQNyGMflSbSDQroz7FEMZiFpmvoUBZ3yNV3+pOqTRLAx1NiPVMZsIbwc96Vv3gxF6O9Iu2aCfRiEKCziMYs624rhDzhLuj3bOYn3gJj1wULuUxjqXJSJ5/f/rmrMiV/d0ykDptPKSVJJj+yDk7Jast0tPkKRzCJISl9Cnp8tchzoCGCRKlUcfkel5CmYJ/Ajx9cnbBBB2QggNRQST3112BI9tCeRQzL5d7vKr5G6mGLtgfN8qdkliZzi66wosImmQ6GAWbvM1WN76ma3LGz8xs4+8t13LJg6GxttBs3N/rxwchuAlL0W8WsMokN0Ei3woHsKdwjIePjy/oCdHiUkdf0IEEIucMS8oQSUekOMgC1zCvGSOlLFarldIyhFCRJnJuYpc1slRn+wJfnHSWpYMOZ+VtIAwnpqZyM4oeORBnyDzmpakUGQwrENaqpDKUAmjgs9pGZg1jKOj4OxPGuWyTuQKhQ10MVGU4KeLlDJLSplbXr67TsX3YvfmqbgoqnPK6iUrr6DtQz7a+EsrZBDfSpiS+DQPtKJUEFR9rZSf+iKGd/VT7i2ZNBePiIcauTc4IAFxrvugWYnk3dRQhOotQFRUgqFAWmliifPw13Iu5kBAS8CkybqpWuHEnpIDVBc7s3UCT0rDnIMjLU+pgFqXMCdRX6RzRQAhoRQRulhYCwbNTqLalN0YWSBfovBNELBijXeR+Egv0grDYCZ7QCgoLhhkJGGVdNSqHlE2CQgRqstFzGBU5bzNF1QpqbcvRo5S23TE8nsk7CaKR8IvG1qVjEkVK7KGYM/XO2wi0Dw7spB6rmiGmua8HZJ7Q3G+B2xd/8gkD3uDHU6+LAqmkwSxx2LY84ZZDKjYgoRC7C3i8PNNnyG2o7ooWG9Q/Cf1ONpb3CDLh5app7bqapSYlQlmx4DEOqd6/tqKxq7jGhKFndviLf8xSlSWKIMXUK0lJbCmyLy2+8SAyDruNNAqX8I6VeI2rC1wsJNJsYK0FxLqFzcTp9yuWWsp0yLmDkOny/ErjcACQFnbPEjZaZylWUJWN6fOCArmlGyBZv7Ph7dvQGvOlqPrBBSyrZZX9RlGdl4Kh0c8sy2OOEmr5+JZiwMv8RtW+R8eq59vlPwwNq/8x8Ze1lk6a4bMH4xE6zrN2uGiefy3LtomQSnrM3+5Tpe5bJfpK91rL7gfFeLCTM7PmxFocU7NAtNr0PMUbOaqcXcYOE/rnUxhmtiNDqNdm9PpMfWJVtStG7JLQXVhg49zfqVMIeXlNHc4ItOdcz+D6QBkF5WiCtB2WpCJRg//np/n1MPRRUYdlYI7AECYlrRr0UObKf7t5P0JLMfKInqHv0EcUxssjsMIjouWI6srZQXKpjmkEtHkCNqqqjoYwRiQPhCYEAC39b64FyS+Q6z+XY7FyePRHnfCXv3j+F8f4hfHb96c/AwKrqeD2LYO45iYAWNuP81Pdanm9gl5vnmaF6sStE4HN9P81du37+I3b38++QBwAXQl+qWArdLDP6C4yjwe66t83rx4hRzyJ8Awvz4o78fPSDLbX7+qnOJzUZ1clqHfYvloejFf6rLEEpXeKiAdW6/gbxC2VGOxXCo6QH43gjKbeTw/U/NzHehqTltcy268fW/snbJhBCuSY3FuhLPSSwjmKdwiZYCmx8dmo+sHze5wMmRbiRSo6io1bcVOiDJBaHK6LZutK4XEjqUELUF9DUK7eeiHK9Tnl5yHUlNObeaHE0v2GNqCoq2hfkxuUvNyWyIXufEz1yqiv3lxM6ZnzEbxc2MJtK1oCgmQVwTHDDvp9adJRpNTS+hMNuMcJ2sXcrgdGq81A/PGZ7CH2PRK7TipOFJG2G6awFQoE6tMomAAA4KZH1BFQ0l3OqN+PNg559xNZREptkXA3XFP5itZr9aZP7Lg7ghcNflmCP9CZucBbUYvdHtpOk03+mxKg35V0aSyNDFicsebAcy2VJAvB5InUVK+otaC+RkBGwRJmWOwQuYJd0TFHAFfMbFmI8Lx9tjo7UUkk8TiCDcnt3vjzYt0maW03YRzsrN7umVhu3nRukwo696y1i8Ss7vF+Au8Xp8bbCj2iuVoG+9gvzWoLbLkFVaQeP4DUuxLENB6UvBE15HANg3oLb1NeN+0lPm8r9CPne30ee8pBNNBhvZVUN94HvkugGlyO1fbJ23D2AarPk42/EiWpcqTwHV8zKLpwLg8497/qyuvbVg4Cr5I87c2ih2vl8eaYlBoeiu9BVFh572W9WukjBs6bXzQPYkDYZ5CeiApSbSgbnX/LgswVmJUnkbZIGhe9O+CylN2sIGHJwy3e5fvKaJAMA7U9YOheGCAOIsiA3wAfKh7lw+oFXuzvZ+4jf6PfJi5Qb52tLfAvmtOEGcy1yTb+ZmNEJwKtgGC289Kooj9J/8Nn5pqNLuQV3rHHKyikknpwNMBIqW00Io8uxo6nKhc2lAThPb8mJpINpWipONKZGqJ0vyC0ibBM6MoCr1ghqRMJAVtigF8lnJ9aXYh8ILmXzk4elHYkGuDL9TwPyLwYulfRScv9FQIGO6tOqBACysfgh/ocIvuAHL4R3Y4LxyO5ki23WVQ3mU+NkYUF6XJJ7vdNppujKxnel/DMFWU0Iv/Fg42laye3YjAZE4Ww2R06iv3FvbvI4IW613Ms3PbYJlHv41hkyaxsncYxTCZMSl1pc5UzkVskzvyqj+Xf8vZzR2V+qJKASa7ijmTDQAChW5M1VQ3o3+bqx2mM1F6XqVcVYgSqgr7Xgp7Es0TUnd8TQUbzFXRPM2H9d0O04zaKq5ujVA347mO61mgojhuXqDks8kmAUemueCgviA8AIGdWlD5qVzOR5Vfw4LNSi+oZUINK/Ky3Ju0Sod/KR3zsvbOEG0FKwzN5uO0lnwrTDox4RMS5Z2PUPZ/c81Yb2i+E/YKmXjQ1GLwtP0qlypZWS0/TZoU0bFy68y9U68ktpKiBPxINFP2XboOvcc4NRyirJCJDgIm4hGpDdkEtyIGBA8xizoZgXN2KJ8IoiXGn+Ckwd3EI7FpF6ZgnNpbNlO+U6Pbn6YxY1TWVtKkiTTwpsAW+PAnLbRTIueWUrQxdMpgyQT1xFB1ytGCfrGVnPrprCvpXD6rue+4gdb3Ety1CFicdDlEB2a6H+WNlriW5wJEp0X0oab2ycu3TnimyROZP4H99eHlr8ev3r8e2irdToU7i7m5606BPV/WvaxEstwbOVq61TFfgmmbVdjOJK0UjM9sRgBK+2mWo1HVKaGoHJkvP758ffI+fn9y/GrYUNdLc3SpSHu6rbCIRmPqzZjODnUn+cjE48i742VUo23MYZs8V9XDt55torOwGCGhDuif/kJYjJmnJ/SaC4ZTo/ibLLENqSqiHlBslpHnCb9BcKNub8K2XTotCauI1jhOjTczPJIHZq/GHrcuinM+UxDXbj/G0Yg6wM51WSwvpFZNh8j0QNqW2ItXJ6PRbkvAvdi4jWJ7MMRk0kkM3RAr7EbZJOiOeNSY9QbobYJyfoXwQp9Jo/gkPkAB/ng0Go1Pv9mQ+VJit4/DQ/79DjPHnRW7E03wMjSvwi9vbiurpFAmaTKX7RBdCMSNX7ffk4Xl/WxyOjCX/sgKm5DR3ANs+Lb3CL9se0vP+Ja32ZKBZmfZ6362seWqPDOFPIO7OuiSeOv2KNZQ0hCgUFkYx05P5Ns/yUo7HFFaq5UO+PZM3o0K7FXduT8doEiQESxCv+EFL0GHkJySEMZe8szhmLS/WsNvGaWgbIVvEeSOtZ7DJfec5mvl79/cBFjjD8cMZHIwPoX/5nNhUGu71ILd43RgQbvJHjZ6S91gKnu2lSnW35pDHxoaRaOb+5yJbV58+Iao80cjjz2c2/TK3+TS7CZMWsmRpVLivXl3Y8ODt4tN7mFsnJ11VuRLdy7T99kiSHNzy4yz+SEVEAtZCXcyGPpOwCLnw41eT/sPMtzQTOn1lrsq94olfxYt/j0DPq9IvjKGtLAaBSd4FbQ4CW47qB2K/W3hzaazp64gqxrHdu9Q1CwwueoRlY5Bm7velZWyU5kVRRZsEGbeEai6o7kEru6Aa1a4AuXWkMLXmhfCFTzGu8Ux1Td0UDZ2xwNU7piT3SZY0Jlu967RbedzQz7LHppbpkNhb/J5t2uG9iIIUdNcHYezt05bXdLXB3B+p97BtFqgKKPeovmSYCi8i/pHu9GIbjiV8fmRfShprHe2VVzQHeuquBiS9DgNP3xsaKEz1CydpzW3nZCXbTmwMp2tz6oqtCXCXVJMqJA8Mu8tWO9uTtVehPCXTUadmuQzMJjXk+o0gt5RtAoRrz0+vV5SZeK1wUn3HkB28HnIVT60GRz0L6edK1U2K1CggQmVB5/DLvmkit0+MItVPBejnlMieBOmY8JTxqek+RyQvOymwchFdUCLhuKzu45W83U0c0EYQXDH3LXdLtAw0sWippuzo00CS/FMYMN7JNZFLaGN5+TojigQDpvzLv+QKqUNYk6iuqCjzyDcflDF8MTzI4sSS8tJekpGNLrt+Kl7naWVXdk0gtItEc4g+v6I4XOOE4T97rk+7+vkl7fTrZwQAZsxcNuO0XSE6WEHVXDbHm3CKqHCZaTXHQ5gUNByenvHVRR4v+AhXV1jZ8J3uY72WqPuXFqkyTnPXrqzdUtac9YehNFK5nQbUPOFerq8EoqHBrB3NZ6aEibfh5oiaVSZuSbV3H6nO7+du4KYHHU+c+KlvMb2CdoPAT77F5sZ+s2YM3M5SzNEVO/CsXdJv/V+DvCdANrr+4/b6/uH3l0A381ykmV5+9xsvrXMyWSPrIb9K9vPDh5PXaDpyoouVTDEjm/GsgOiyHfXGPoiBPY6e7yS/fre16woge2JW1MCzxNXTxTnphsQacN044Ju049R2Mfcv4ANmL1rtfYA5Ly9nxrsDttRthj+joMXIUhT+GlYGJuv7pT5SsjdUaXiBPOE2RC6lGGABy+G3Xv9vKdQS6OJKd3P6m4sGxQQ7o+MTR3t3iLSvBANEBGYawjW925B6W3sVoz7Q7F3YDHetom+vtBXRz19Iav/8sJRtL+paPsb5NKe+oR6IJle0PrYSWifizGjiwetLh5sA1pu5f7Qg/b4Nias5h74mnvwhclPOrOfbJCEvDGmL8TUXbyCukNH3UG4KdAfGkMctQb246aUf2hsZdTS9GOfJKPecZJqcxppfecdzsf6nQPnfPhxZ5+fH7MjohJaIXnYQzpOjqq1d31Pgz8Mu6TfYvL6FpvXntEbYlie7kParcbds2TORyr1eyOHWZpzKWPxrtIcMX5Znx0dhn6GyOjsgYnMKTp58WIy3j31GUPa6fHGlUdASCdmKVIZk9DYL6N6TCSVorvQoEfl9utQ/l64Uqvik2ovf5vvnRgwfd9kvgabaRb5ZRB2UhomhBY8o13e48jWYiQQa7DC16epOcrcghB5oQV9dowhLkT4BjW7LZV0AyJl/eY+tv0kanGz6casctIpb9zkEFtNehdWsztyZvOD5ULeU91298i5dIuYg0Z7+aNjb/aTJy6Szf5EPCz5ja8+ujtOYqWPBNrEivkWS/lJWfnTLVPWCC0CyM9Lwbix1v+CulX2+b15eXw3L9SY6/Ez72twQ3KXIW04Mtft+JtdwxEVCLuHVDXqLUpTciq5ze8j0B4Yn8Rq4jwKpcT2Kwh2bttzx653/dF5112G189tTHaJDL1Fyjzz14BiLrW6O8M02XmSro52dsOhXyAePt4ABDmA5hUnmS6zpLg2uLn5X1yVvJE=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
